In [295]:
from mpmath import coulombf, coulombg
import numpy as np
import mpmath
import numba
from scipy.integrate import simpson
from scipy.linalg import eigh
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar
from scipy.special import genlaguerre, gamma
from scipy.integrate import quad
from scipy.constants import hbar
# 设置高精度计算（50位小数）
mpmath.mp.dps = 50
# # 计算 F 和 G
# F_values = [mpmath.coulombf(l, eta, rho) for rho in rho_values]
# G_values = [mpmath.coulombg(l, eta, rho) for rho in rho_values]

In [296]:
e2=1.43997 ; hbarc=197.3269718 ; amu=931.49432
z1=82 ; m1=207*amu-22.451968
z2=2 ; m2=4*amu+2.42491587
Q=7.59456013
mu=m1*m2/(m1+m2)
v0=162.3 ; a0=0.4 ; r0=7.642; l=5; P0=0.03
rc=r0
k=np.sqrt(2*mu*Q/(hbarc**2))
eta= z1 * z2 * e2 * mu / (hbarc**2 * k) 
# z1=1 ; m1=1.0078*amu
# z2=0 ; m2=1.0087*amu
# l=0
# mu=m1*m2/(m1+m2)

In [297]:
#potential function
@numba.njit
def wspot(r,v0,a0,r0):
    return -v0*(1+np.cosh(r0/a0))/(np.cosh(r/a0)+np.cosh(r0/a0))

@numba.njit
def vc(r,z1,z2,rc):
    return np.where(
        r < rc,
        z1 * z2 * e2 * (3 - r**2 / rc**2) /(2 * rc),
        z1 * z2 * e2 / r
    )
    
@numba.njit
def vpot(r,v0,a0,r0,z1,z2,rc,l):
    return wspot(r,v0,a0,r0)+vc(r,z1,z2,rc)

#coulomb function
def F_L(r, k, eta, L):
    """库仑函数 F_L(η, kr)"""
    return coulombf(L, eta, k * r)

def G_L(r, k, eta, L):
    """库仑函数 G_L(η, kr)"""
    return coulombg(L, eta, k * r)

In [298]:
def isotropic_ho_radial(n, l, r, alpha=1.0):
    # 计算归一化常数
    norm = np.sqrt(2 * alpha**(3/2) * gamma(n+1) / gamma(n + l + 3/2))
    
    # 计算径向部分
    xi = alpha * r**2
    laguerre = genlaguerre(n, l + 0.5)(xi)
    
    # 组合所有部分
    radial_wave = norm * (alpha**0.5 * r)**l * np.exp(-xi/2) * laguerre
    
    return radial_wave

# r*R_nl(r)
def phi_basis(n, l, r, alpha=1.0):
    
    return r * isotropic_ho_radial(n, l, r, alpha)

In [299]:
h=0.01
r = np.arange(1e-8, 40, h)

In [300]:
# 计算FL和VV,FL是coulomb函数的离散值,VV是势函数的离散值,被用于进行DWBA的积分步骤
FL = np.array([F_L(ri, k, eta, l) for ri in r], dtype=np.complex128)

In [301]:
def phi_basis_second_derivative(n, l, r, alpha=1.0, dr=h):
    """
    计算φ_{nl}(r)对r的二阶导数
    内部点使用五点中心差分法
    边界点仅使用phi数组中的值计算
    """
    phi = phi_basis(n, l, r, alpha)
    
    d2phi = np.zeros_like(phi)
    
    # 内部点使用五点中心差分法
    d2phi[2:-2] = (-phi[4:] + 16*phi[3:-1] - 30*phi[2:-2] 
                   + 16*phi[1:-3] - phi[:-4]) / (12*dr**2)
    
    # 边界点简单使用三点差分法（仅用phi数组）
    d2phi[0] = (phi[0] - 2*phi[1] + phi[2]) / (dr**2)  
    d2phi[1] = (phi[0] - 2*phi[1] + phi[2]) / (dr**2)  
    
    d2phi[-1] = (phi[-3] - 2*phi[-2] + phi[-1]) / (dr**2)  
    d2phi[-2] = (phi[-3] - 2*phi[-2] + phi[-1]) / (dr**2)  
    
    return d2phi

In [302]:
# def numerov for u(0):(y(0))

def u_zero(v0, a0, r0, rc, l, k, eta, E):
    
    z = np.zeros_like(r, dtype=np.complex128)
    y = np.zeros_like(r, dtype=np.complex128)

    # 边界条件
    z[-1] = G_L(r[-1], k, eta, l) + 1j * F_L(r[-1], k, eta, l)
    z[-2] = G_L(r[-2], k, eta, l) + 1j * F_L(r[-2], k, eta, l)

    f = 2*mu/hbarc**2 * (vpot(r, v0, a0, r0, z1, z2, rc, l)+(hbarc**2/2/mu)*l*(l+1)/r**2 - Q) 

    # Numerov方法
    for i in range(len(z)-2, 0, -1):
        z[i-1] = 2*z[i] - z[i+1] + h**2 * f[i] * z[i]

    # 计算trial wavefunction y
    y = (1 - h**2 * f / 12) * z
    y = np.array(y, dtype=np.complex128)
    
    # 归一化
    y_squared = np.abs(y)**2
    norm_factor = np.sqrt(simpson(y=y_squared, x=r))
    y /= norm_factor


    return np.real(y[0])


In [303]:
ndim = 50
H = np.zeros((ndim, ndim), dtype=np.complex128)


In [304]:
def Model(m1,m2,z1,z2,l,rc,r0,v0,a0,P0):
# H_ij
    for i in range(ndim):
        for j in range(ndim):
            phi_i = phi_basis(i, l, r, alpha=0.5)
            phi_j = phi_basis(j, l, r, alpha=0.5)
            d2phi_j = phi_basis_second_derivative(j, l, r, alpha=0.5, dr=h)
            kinetic_term = (-hbarc**2/2/mu) * simpson(phi_i.conj() * d2phi_j , x=r)
            centrifugal_term = (hbarc**2/2/mu)*l * (l + 1) * simpson(phi_i.conj() * phi_j / r**2 , x=r)
            vpot_term = simpson(phi_i.conj() * vpot(r,v0,a0,r0,z1,z2,rc,l)*phi_j,x=r)
            H[i, j]=kinetic_term+vpot_term+centrifugal_term

#eig(H)
    eigenvalues, eigenvectors = eigh(H)

# E_min(positive)=Q
    Q = min([ev for ev in eigenvalues if ev > 0])

#Q\pm 0.5 search with BC u(0)=0

    try:
        result = root_scalar(
            lambda E: u_zero(v0, a0, r0, rc, l, np.sqrt(2*mu*abs(E)/hbarc**2), 
                           z1*z2*e2*mu/(hbarc**2), E),
            bracket=[Q-0.5, Q+0.5],
            method='brentq'
        )
        Q0 = result.root
        print(f"Found Q0: {Q0:.6f} MeV")
    except:
        Q0 = Q  
        print("Root finding failed, using Q as Q0")
    
    z1 = np.zeros_like(r, dtype=np.complex128)
    y1 = np.zeros_like(r, dtype=np.complex128)   

    # 边界条件
    z1[-1] = G_L(r[-1], k, eta, l) + 1j * F_L(r[-1], k, eta, l)
    z1[-2] = G_L(r[-2], k, eta, l) + 1j * F_L(r[-2], k, eta, l)
    f1 = 2*mu/hbarc**2 * (vpot(r, v0, a0, r0, z1, z2, rc, l)+ (hbarc**2/2/mu)*l*(l+1)/r**2 - Q0)    
    # Numerov方法
    for i in range(len(z1)-2, 0, -1):
        z1[i-1] = 2*z1[i] - z1[i+1] + h**2 * f1[i] * z1[i]
    
    y1 = (1 - h**2 * f1 / 12) * z1
    y1 = np.array(y1, dtype=np.complex128)
    
    # 归一化
    y1_squared = np.abs(y1)**2
    norm_factor = np.sqrt(simpson(y=y1_squared, x=r))
    y1 /= norm_factor

    # 计算FL和VV,FL是coulomb函数的离散值,VV是势函数的离散值,被用于进行DWBA的积分步骤
    VV = vpot(r, v0, a0, r0, z1, z2, rc, l)-z1*z2*e2/r
    # 计算积分结果
    result = simpson(y=VV*FL*y1, x=r)
    
    # 计算半衰期
    Gamma = P0 * abs(result)**2 * 4*mu/hbarc**2/k
    T_half = hbarc * np.log(2) / Gamma * 1e-23/3

    return Q0, T_half


qq,tt=Model(m1,m2,z1,z2,l,rc,r0,v0,a0,P0)
print(qq,tt)


Root finding failed, using Q as Q0
7.262013771090679 8.129654477755464e+39


### Def prior and posterior

In [305]:
# y=[8.8624,1.68e-3]  #Q,T_half
# sigma_Q=2.3e-3      #MeV
# sigma_t=0.01e-3     #s
# sigma=[sigma_Q,sigma_t]
# #a=[v0,a0,r0,P]
# def log_prior(a):
#     vv,aa,rr,pp=a
#     R=1
#     v0=162.3; a0=0.4; r0=7.660; P0=0.03
    
#     sigma_v=3*R
#     sigma_a=0.1*R
#     sigma_r=0.1*R
#     sigma_P=0.01*R

#     prior_v=-0.5*(vv-v0)**2/sigma_v**2
#     prior_a=-0.5*(aa-a0)**2/sigma_a**2
#     prior_r=-0.5*(rr-r0)**2/sigma_r**2
#     prior_P=-0.5*(pp-P0)**2/sigma_P**2

#     return prior_v+prior_a+prior_r+prior_P

# #a=[v0,a0,r0,P]
# def log_posterior(a,y,sigma,m1,m2,z1,z2,l):
#     log_prior_value=log_prior(a)
#     q,t=Model(m1,m2,z1,z2,l,a[2],a[2],a[0],a[1],a[3])
#     log_likelihood=-0.5*((q-y[0])**2/sigma[0]**2+(t-y[1])**2/sigma[1]**2)
#     return log_prior_value+log_likelihood

In [306]:
# M=4
# nwalkers=2*M
# initial_pos = [v0, a0, r0, P0]
# pertubation_scale=[1, 0.05, 0.1, 0.001]
# a=np.array([initial_pos+np.random.normal(0,pertubation_scale,M) for _ in range(nwalkers)])
# print(a)

# import emcee
# import multiprocessing
# with multiprocessing.Pool() as pool:
#     sampler = emcee.EnsembleSampler(nwalkers, M, log_posterior, args=[y,sigma,z1,z2,l,Q], a=0.2, pool=pool)
#     state = sampler.run_mcmc(a, 200)
#     sampler.reset()
#     sampler.run_mcmc(state, 2000)

In [307]:
# import prettyplease

# samples = sampler.get_chain(flat=True)

# labels=["$V_0$","$a_0","$r_0$","$P_0$"]
# fig = prettyplease.corner(samples, labels=labels)
# plt.show()